# Step 2: Build Cycle-Level Data

This notebook converts the already-preprocessed row-level train/validation/test files into cycle-level data for sequence-model training.

Why: for our sequence-model setup, one time step should mean one battery cycle, not one raw measurement row.

Important: this notebook does not refit scaling. Scaling was already done in the shared preprocessing step on the `models` branch.

In [ ]:
# We use the already-preprocessed files so GRU and GLU start from the same shared split.
from pathlib import Path

import pandas as pd

current_dir = Path.cwd().resolve()
if (current_dir / "data" / "processed" / "train_dataset.csv").exists():
    PROJECT_ROOT = current_dir
else:
    PROJECT_ROOT = (current_dir / "..").resolve()

TRAIN_FILE = PROJECT_ROOT / "data" / "processed" / "train_dataset.csv"
VALIDATION_FILE = PROJECT_ROOT / "data" / "processed" / "validation_dataset.csv"
TEST_FILE = PROJECT_ROOT / "data" / "processed" / "test_dataset.csv"

CYCLE_OUTPUT = PROJECT_ROOT / "data" / "processed" / "cycle_processed_data.csv"
CYCLE_TRAIN_OUTPUT = PROJECT_ROOT / "data" / "processed" / "cycle_train_dataset.csv"
CYCLE_VALIDATION_OUTPUT = PROJECT_ROOT / "data" / "processed" / "cycle_validation_dataset.csv"
CYCLE_TEST_OUTPUT = PROJECT_ROOT / "data" / "processed" / "cycle_test_dataset.csv"

print("Project root:", PROJECT_ROOT)
print("Train file:", TRAIN_FILE)
print("Validation file:", VALIDATION_FILE)
print("Test file:", TEST_FILE)

In [ ]:
# Load the preprocessed train, validation, and test datasets.
train_df = pd.read_csv(TRAIN_FILE)
validation_df = pd.read_csv(VALIDATION_FILE)
test_df = pd.read_csv(TEST_FILE)

print("Train shape:", train_df.shape)
print("Validation shape:", validation_df.shape)
print("Test shape:", test_df.shape)
print("Train batteries:", sorted(train_df["battery_id"].unique()))
print("Validation batteries:", sorted(validation_df["battery_id"].unique()))
print("Test batteries:", sorted(test_df["battery_id"].unique()))

train_df.head()

In [ ]:
# Define model features and non-feature columns.
# cycle is needed for ordering, but it must not be passed into the model as an input feature.
MODEL_FEATURE_COLUMNS = ["chI", "chV", "chT", "disI", "disV", "BCt", "SOH"]
TARGET_COLUMN = "RUL"
ID_COLUMN = "battery_id"
ORDER_COLUMN = "cycle"

required_columns = [ORDER_COLUMN, *MODEL_FEATURE_COLUMNS, ID_COLUMN, TARGET_COLUMN]

for name, df in [("train", train_df), ("validation", validation_df), ("test", test_df)]:
    missing_columns = [column for column in required_columns if column not in df.columns]
    if missing_columns:
        raise ValueError(f"{name} data is missing required columns: {missing_columns}")

print("Model input features:", MODEL_FEATURE_COLUMNS)
print("Kept for sorting/grouping, not model input:", [ID_COLUMN, ORDER_COLUMN])
print("Target:", TARGET_COLUMN)

In [ ]:
#  Define a reusable function to aggregate rows into one row per battery-cycle.
#  each cycle has many measurement rows, but our sequence model needs one feature vector per cycle.
def build_cycle_level_dataset(df):
    aggregation_rules = {column: "mean" for column in MODEL_FEATURE_COLUMNS}
    aggregation_rules[TARGET_COLUMN] = "mean"

    cycle_df = (
        df[required_columns]
        .groupby([ID_COLUMN, ORDER_COLUMN], as_index=False)
        .agg(aggregation_rules)
        .sort_values([ID_COLUMN, ORDER_COLUMN])
        .reset_index(drop=True)
    )

    cycle_df[TARGET_COLUMN] = cycle_df[TARGET_COLUMN].round().astype(int)
    return cycle_df[[ORDER_COLUMN, *MODEL_FEATURE_COLUMNS, ID_COLUMN, TARGET_COLUMN]]

In [ ]:
# Build cycle-level train, validation, test, and combined datasets.
# we aggregate each split separately so no battery information is mixed.
cycle_train_df = build_cycle_level_dataset(train_df)
cycle_validation_df = build_cycle_level_dataset(validation_df)
cycle_test_df = build_cycle_level_dataset(test_df)
cycle_df = (
    pd.concat([cycle_train_df, cycle_validation_df, cycle_test_df], ignore_index=True)
    .sort_values([ID_COLUMN, ORDER_COLUMN])
    .reset_index(drop=True)
)

print("Cycle-level train shape:", cycle_train_df.shape)
print("Cycle-level validation shape:", cycle_validation_df.shape)
print("Cycle-level test shape:", cycle_test_df.shape)
print("Cycle-level combined shape:", cycle_df.shape)

cycle_df.head()

In [ ]:
# Validate the cycle-level data.
# one duplicate battery-cycle would mean one time step is not uniquely defined.
duplicate_count = cycle_df.duplicated(subset=[ID_COLUMN, ORDER_COLUMN]).sum()
if duplicate_count != 0:
    raise ValueError(f"Found {duplicate_count} duplicate battery-cycle rows")

summary = cycle_df.groupby(ID_COLUMN).agg(
    rows=(ORDER_COLUMN, "count"),
    min_cycle=(ORDER_COLUMN, "min"),
    max_cycle=(ORDER_COLUMN, "max"),
    min_RUL=(TARGET_COLUMN, "min"),
    max_RUL=(TARGET_COLUMN, "max"),
)

summary

In [ ]:
# Confirm there is still no battery overlap between splits.
# if the same battery appears in multiple splits, evaluation results would be misleading.
split_batteries = {
    "train": set(cycle_train_df[ID_COLUMN].unique()),
    "validation": set(cycle_validation_df[ID_COLUMN].unique()),
    "test": set(cycle_test_df[ID_COLUMN].unique()),
}

split_names = list(split_batteries)
for left_index, left_name in enumerate(split_names):
    for right_name in split_names[left_index + 1:]:
        overlap = split_batteries[left_name] & split_batteries[right_name]
        if overlap:
            raise ValueError(f"{left_name}/{right_name} battery overlap found: {sorted(overlap)}")

print("Train batteries:", sorted(split_batteries["train"]))
print("Validation batteries:", sorted(split_batteries["validation"]))
print("Test batteries:", sorted(split_batteries["test"]))

In [ ]:
# Save the cycle-level datasets.
cycle_df.to_csv(CYCLE_OUTPUT, index=False)
cycle_train_df.to_csv(CYCLE_TRAIN_OUTPUT, index=False)
cycle_validation_df.to_csv(CYCLE_VALIDATION_OUTPUT, index=False)
cycle_test_df.to_csv(CYCLE_TEST_OUTPUT, index=False)

print("Saved full cycle-level dataset to:", CYCLE_OUTPUT)
print("Saved cycle-level train dataset to:", CYCLE_TRAIN_OUTPUT)
print("Saved cycle-level validation dataset to:", CYCLE_VALIDATION_OUTPUT)
print("Saved cycle-level test dataset to:", CYCLE_TEST_OUTPUT)

In [ ]:
# Confirm the final columns before moving to window creation.
# The model input should include only feature columns; cycle, battery_id, and RUL have special roles.
print("Cycle-level columns:", list(cycle_df.columns))
print("Model input features:", MODEL_FEATURE_COLUMNS)
print("Used for sorting/window grouping:", [ID_COLUMN, ORDER_COLUMN])
print("Prediction target:", TARGET_COLUMN)

cycle_df.head()